In [3]:
from IPython.display import Markdown , display 

display(Markdown("## 💥Data Preparation... For Research AI Agent")) 



## 💥Data Preparation... For Research AI Agent

In [ ]:
import json 
import logging
import random 
from pathlib import Path 
from datasets import load_dataset 
import os
from tqdm import tqdm 

OPENHERMES = "teknium/OpenHermes-2.5" 
ALPACA = "yahma/alpaca-cleaned" 
NOROBOT="HuggingFaceH4/no_robots" 

log = logging.getLogger(__name__)
class Data_Preparation: 
    def __init__(self):
        self.output_path = Path("dataset/d1_instruction.jsonl")  
        os.makedirs(self.output_path.parent, exist_ok=True)

    def collect(self): 
        samples = []
        try: 
            print("Loading the OPENHERMES Dataset....")  
            display(Markdown("#### 🔽 Steps ... "))
            ds = load_dataset(OPENHERMES , split="train" , streaming=True)  
            count = 0
            for row in tqdm(ds ,total=20000 ,  desc="OpenHermes Processing"): 
                if count >= 20000: 
                    break 
                if row.get("conversations"): 
                    sample = self._parse_opnhermes(row)   
                    if sample: 
                        samples.append(sample) 
                        count+=1   
                    
            log.info(f"Successfully Loaded the {count} data")
        except Exception as e: 
            log.warning(f"error : {e} aa gaya bhai during processing the openhermes")

        try: 
            print("Loading the ALPACA Dataset....") 
            ds = load_dataset(ALPACA , split="train") 
            c = 0   
            for row in tqdm(ds , desc="Processing Alpaca...."): 
                sample = self._parse_alpaca(row)   
                if sample: 
                    samples.append(sample)  
                    c+=1
                    
                        
            log.info(f"From Alpaca we loaded {c}k rows of data")
        except Exception as e: 
            log.warning(f"error {e} aa gaya bhai during processing the Alpaca") 

        try: 
            print("Loading the no_robots Dataset....") 
            ds = load_dataset(NOROBOT , split="train")  
            count = 0
            for row in tqdm(ds , desc="Processing no_robot.."): 
                sample = self._parse_norobot(row)   
                if sample: 
                    samples.append(sample) 
                    count+=1 
            
            log.info(f"Successfully Loaded the {count} data")
        except Exception as e: 
            log.warning(f"error aa gaya bhai during processing the no_robot error : {e}")

        random.shuffle(samples) 
        self._write(samples) 
        log.info(f"✅ Dataset 1 saved: {len(samples)} samples → {self.output_path}")
        return 

    def _parse_opnhermes(self , row:dict): 
        convs = row.get("conversations" , []) 
        if len(convs)<2: 
            None 

        system = "" 
        messages = [] 
        for row in convs: 
            role = row.get("from" , "").strip()
            value =  row.get("value" , "").strip() 

            if not value: 
                continue 

            if role=="system": 
                system=value
            elif role in ["human" , "user"]: 
                messages.append({"role":"user" , "content":value}) 
            elif role in ['gpt',"assistant"]: 
                messages.append({"role":"assistant" , "content":value})  
        if not messages: 
            return   None 
        
        return {
            "dataset": "openhermes" , 
            "task": "instruction_tune" , 
            "system": system or "You Are Helpful assistant." , 
            "messages":messages 
        }
 
    def _parse_alpaca(self , row:dict): 
        instruction = row.get("instruction" , "").strip() 
        input_text = row.get("input" , "").strip()
        response = row.get("output" , "").strip() 

        if not instruction or not response: 
            return None 

        if len(response) <4: 
            return None 

        user_content = instruction 
        if input_text: 
            user_content = f"{instruction}\n\ninput:\n{input_text}" 

        return {
            "dataset":"alpaca" , 
            "task" : "instruction_tune" , 
            "system" : "You are Helpful assistant." , 
            "messages":[
                {"role":"user" , "content":user_content} , 
                {"role":"assistant" , "content":response}
            ]
        } 
 
    def _parse_norobot(self , row): 
        messages = row.get("messages","") 
        if not messages: 
            return None 

        system = "" 
        message_content = []
        for val in messages: 
            role = val.get("role" , "") 
            value = val.get("content" , "") 

            if role=="system": 
                system = value 
            elif role=="user": 
                message_content.append({"role":"user" , "content":value})
            elif role in ("system" , "assistant"): 
                message_content.append({"role":"assistant" , "content":value}) 
        
        if len(message_content)<2:
            return None 

        return {
            "dataset":"no_robot" , 
            "task":"instruction_tune" , 
            "system" : system or "You are Helpful assistant." , 
            "messages" : message_content
         }


    def _write(self , sample:list): 
        with open(self.output_path , "w") as f: 
            for s in sample: 
                f.write(json.dumps(s , ensure_ascii=False)+"\n") 

        log.info(f"all data is written to {self.output_path}")
 

In [33]:
import shutil
import os

if os.path.isdir("dataset/d1_instruction.jsonl"):
    shutil.rmtree("dataset/d1_instruction.jsonl")

In [34]:
obj = Data_Preparation() 
obj.collect()

Loading the OPENHERMES Dataset....


#### 🔽 Steps ... 

OpenHermes Processing: 100%|██████████| 20000/20000 [16:41<00:00, 19.97it/s]


Loading the ALPACA Dataset....


Processing Alpaca....: 100%|██████████| 51760/51760 [00:01<00:00, 31703.77it/s]


Loading the no_robots Dataset....


Processing no_robot..: 100%|██████████| 9500/9500 [00:00<00:00, 15516.16it/s]


In [ ]:
import json

data = []

with open("dataset/d1_instruction.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

data[:2]

[{'dataset': 'openhermes',
  'task': 'instruction_tune',
  'system': 'You Are Helpful assistant.',
  'messages': [{'role': 'user',
    'content': 'Determine whether the following series converges or diverges: Σ(1/n^2) from n=1 to infinity. Explain your reasoning.'},
   {'role': 'assistant',
    'content': 'The series Σ(1/n^2) from n=1 to infinity converges. This is because it is a p-series with p = 2, which is greater than 1. In general, a p-series Σ(1/n^p) converges if p > 1 and diverges if p ≤ 1. This result can be proven using the integral test or by comparing the series to another known convergent series.\n\nIn this case, since p = 2 > 1, the series converges. This particular series is also known as the Basel problem, and its sum converges to (π^2)/6.'}]},
 {'dataset': 'alpaca',
  'task': 'instruction_tune',
  'system': 'You are Helpful assistant.',
  'messages': [{'role': 'user',
    'content': 'Describe the setting of an enchanted forest.'},
   {'role': 'assistant',
    'content'

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset/d1_instruction.jsonl"
)

dataset.push_to_hub("Rohit-Katkar2003/instruction-dataset")

: 

## Summarization dataset

In [ ]:
SUMMARY_PROMPTS =  [
    "Summarize the following text concisely:\n\n{text}",
    "Extract the key findings and main points from this text:\n\n{text}",
    "Write a brief, accurate summary suitable for a research report:\n\n{text}",
    "What are the most important takeaways from this content?\n\n{text}",
    "Provide a dense, informative summary of the following:\n\n{text}",
]

SYSTEM_PROMPT = (
    "You are a research summarization assistant. "
    "Your summaries are accurate, concise, and preserve all key facts."
)

class Prep_Summarization_data: 
    def __init__(self): 
        self.output_path = Path("dataset/d2_summarization.jsonl") 
        os.makedirs(self.output_path.parent , exist_ok=True) 

    def collect(self): 
        display(Markdown("## 🎈Loading Summarization dataset....")) 

        try: 
            count = 40000 ## taking 40000 rows only 
            ds = load_dataset("abisee/cnn_dailymail" , split="train" , streaming=True)  
            indices = random.sample(range(len(ds)) , min(20000 , len(ds)))  # it return list
            for i in indices: 
                 
                
        except Exception as e: 
            print((f"got error : {e}"))
            log.warning(f"got error : {e}")

### Extraction dataset
- it help us to extract important relation from data which helper research better and find complex relations

In [24]:
import json 
import logging 
import random 
from pathlib import Path 
from datasets import load_dataset  
import os 
from tqdm import tqdm 
log = logging.getLogger(__name__) 


SYSTEM_PROMPT = (
    "You are an information extraction agent. "
    "Extract precise facts from text. "
    "Return valid JSON. If the answer cannot be found, return null for that field."
)

EXTRACTION_PROMPTS = [
    "Extract the answer to this question from the provided context. Return JSON.\n\nContext:\n{context}\n\nQuestion: {question}",
    "Given this source text, find the specific information requested. Return as structured JSON.\n\nSource:\n{context}\n\nWhat to find: {question}",
    "Analyze this text and extract the requested fact. If not present, state null.\n\nText:\n{context}\n\nFact to extract: {question}",
]

RELATION_PROMPTS = [
    "Extract all entity relationships from this text as JSON triplets (subject, predicate, object).\n\nText: {text}",
    "Identify the key factual relationships in this text. Return as JSON array of {{subject, relation, object}}.\n\nText: {text}",
]

class Extraction_data_prep: 
    def __init__(self):
        self.output_path = Path("Extraction/Extraction.jsonl") 
        os.makedirs(self.output_path.parent , exist_ok=True) 

    def collect(self): 
        samples = [] 
        max_samples = 50000
        try:  
            count = 0
            ds = load_dataset("rajpurkar/squad_v2" , split="train") 
            for row in tqdm(ds , desc="Squad_v2 data loading ... " , total=max_samples):  
                if count>=max_samples: 
                    break
                sample = self._parse_squad(row) 
                if sample: 
                    samples.append(sample) 
                    count += 1  
        except Exception as e: 
            log.error(f"got error : {e}")  
        
        try:
            ds = load_dataset("google-research-datasets/natural_questions",
                              "default", split="train", streaming=True)
            count = 0
            for row in tqdm(ds , desc="Loading natural question dataset .........." , total=max_samples):
                if count >= max_samples:
                    break
                sample = self._parse_natural_questions(row)
                if sample:
                    samples.append(sample)
                    count += 1
            log.info(f"Loaded {count} NaturalQ samples")
        except Exception as e:
            log.warning(f"  NQ failed ({e}), trying TriviaQA fallback ...")
            
        
        random.shuffle(samples)
        self._write(samples)
        log.info(f"✅ Dataset 3 saved: {len(samples)} samples → {self.output_path}")


        
    def _parse_squad(self , row:dict) :  
        # print(row)
        context = row.get("context" , "") 
        answer = row.get("answers" , "") 
        question = row.get("question" ,"") 
        
        if not context or not question:
            return None

        answer_texts = answer.get("text", []) 
        if answer_texts:
            answer = answer_texts[0]
            output = json.dumps({
                "answer": answer,
                "found": True,
                "confidence": "high",
            })
        else:
            output = json.dumps({
                "answer": None,
                "found": False,
                "confidence": "N/A",
            })

        prompt_template = random.choice(EXTRACTION_PROMPTS)
        user_content = prompt_template.format(context=context, question=question)

        return {
            "dataset": "squad_v2",
            "task": "information_extraction",
            "system": SYSTEM_PROMPT,
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": output},
            ],
        } 
    
    def _parse_natural_questions(self, row: dict):
       
        try:
            question = row["question"]["text"].strip()
            annotations = row.get("annotations", {})
            short_answers = annotations.get("short_answers", [[]])[0]

            if not question:
                return None

            
            yes_no = annotations.get("yes_no_answer", ["NONE"])[0]
            if yes_no in ("YES", "NO"):
                answer = yes_no.lower()
            elif short_answers and short_answers.get("text"):
                answer = short_answers["text"][0]
            else:
                answer = None

            output = json.dumps({
                "answer": answer,
                "found": answer is not None,
                "confidence": "high" if answer else "N/A",
            })

            user_content = f"Answer this factual question. Return JSON.\n\nQuestion: {question}"

            return {
                "dataset": "natural_questions",
                "task": "information_extraction",
                "system": SYSTEM_PROMPT,
                "messages": [
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": output},
                ],
            }
        except Exception:
            return None


    def _write(self, samples: list):
        with open(self.output_path, "w") as f:
            for s in samples:
                f.write(json.dumps(s, ensure_ascii=False) + "\n")



In [25]:

obj = Extraction_data_prep() 
obj.collect()

Squad_v2 data loading ... : 100%|██████████| 50000/50000 [00:03<00:00, 12522.12it/s]


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Loading natural question dataset ..........: 100%|██████████| 50000/50000 [22:51<00:00, 36.45it/s]  


In [ ]:
from huggingface_hub import HfApi 

api = HfApi() 

api.create_repo(
    repo_id="Rohit-Katkar2003/Extractional-dataset",
    repo_type="dataset",
    exist_ok=True
)

api.upload_folder(
    folder_path="Extraction",
    repo_id="Rohit-Katkar2003/Extractional-dataset",
    repo_type="dataset",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Rohit-Katkar2003/Extractional-dataset/commit/b60dea5b2057aece334cac5cb880782f78bf472a', commit_message='Upload folder using huggingface_hub', commit_description='', oid='b60dea5b2057aece334cac5cb880782f78bf472a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Rohit-Katkar2003/Extractional-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Rohit-Katkar2003/Extractional-dataset'), pr_revision=None, pr_num=None)

In [28]:

import json
import logging
import os
import random
import re
from itertools import product
from pathlib import Path

log = logging.getLogger(__name__)

SYSTEM_PROMPT = (
    "You are a research planning agent. "
    "Break complex research questions into specific, searchable sub-queries. "
    "Return valid JSON only — no preamble, no explanation."
)

# ─────────────────────────────────────────────────────────────────────────────
# SEED QUESTIONS by domain
# These are carefully crafted to cover the range of research agent use cases
# ─────────────────────────────────────────────────────────────────────────────
SEED_QUESTIONS = {
    "technology": [
        "What are the latest breakthroughs in quantum computing?",
        "How is generative AI changing software development workflows?",
        "What are the security vulnerabilities in large language models?",
        "How do transformer architectures compare to state space models?",
        "What is the current state of autonomous vehicle technology?",
        "How are edge computing and IoT converging in 2024?",
        "What are the most impactful open-source AI frameworks right now?",
        "How is WebAssembly changing backend development?",
        "What are the main challenges in federated learning deployments?",
        "How does Rust compare to C++ for systems programming today?",
    ],
    "science": [
        "What are the latest discoveries in CRISPR gene editing?",
        "How close are we to a practical nuclear fusion reactor?",
        "What recent findings challenge our understanding of dark matter?",
        "How is microbiome research changing our approach to mental health?",
        "What are the emerging treatments for Alzheimer's disease?",
        "How is climate change affecting ocean biodiversity?",
        "What breakthroughs have occurred in room-temperature superconductors?",
        "How are mRNA vaccines being developed beyond COVID?",
        "What is the current status of Mars colonization research?",
        "How do epigenetic changes affect aging processes?",
    ],
    "business": [
        "What is the current state of the electric vehicle market?",
        "How are central banks responding to persistent inflation?",
        "What is driving the consolidation in the semiconductor industry?",
        "How is remote work changing commercial real estate?",
        "What are the most successful AI startup business models?",
        "How are ESG regulations affecting investment strategies?",
        "What is the impact of deglobalization on supply chains?",
        "How are subscription business models evolving post-pandemic?",
        "What are the competitive dynamics in cloud computing?",
        "How is fintech disrupting traditional banking in emerging markets?",
    ],
    "academic": [
        "What are the most cited papers on attention mechanisms in 2024?",
        "How has the replication crisis affected psychology research?",
        "What methodologies are emerging in computational social science?",
        "How are large language models being used in drug discovery research?",
        "What are the leading theories on consciousness in neuroscience?",
        "How is meta-analysis methodology evolving in medical research?",
        "What are the most influential economics papers of the last decade?",
        "How is interpretability research advancing in deep learning?",
        "What are the key debates in philosophy of mind about AI?",
        "How are researchers measuring AI system alignment?",
    ],
    "current_events": [
        "What are the latest developments in the US-China trade relationship?",
        "How are European countries responding to the energy transition?",
        "What is the current state of cryptocurrency regulation globally?",
        "How are AI regulations evolving across different jurisdictions?",
        "What are the latest geopolitical tensions affecting tech supply chains?",
        "How is the open-source AI movement challenging proprietary models?",
        "What is happening with global water scarcity policy?",
        "How are governments using AI for public services?",
        "What is the current status of space commercialization?",
        "How is social media regulation evolving internationally?",
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
# DECOMPOSITION TEMPLATES
# Rule-based sub-query generation patterns
# ─────────────────────────────────────────────────────────────────────────────
DECOMPOSITION_TEMPLATES = [
    # Pattern: What → Current state + History + Players + Future + Impact
    {
        "pattern": "What are the latest {topic}",
        "sub_queries": [
            "recent developments in {topic_short} 2024 2025",
            "timeline history of {topic_short}",
            "leading researchers organizations in {topic_short}",
            "future predictions {topic_short}",
            "practical applications {topic_short}",
        ],
        "source_types": ["web", "arxiv", "news"],
        "strategy": "breadth_first",
    },
    # Pattern: How → Mechanism + Evidence + Examples + Challenges + Solutions
    {
        "pattern": "How {verb} {topic}",
        "sub_queries": [
            "how does {topic_short} work mechanism",
            "evidence research showing {topic_short} impact",
            "case studies examples {topic_short}",
            "challenges problems with {topic_short}",
            "solutions approaches to {topic_short}",
        ],
        "source_types": ["web", "arxiv"],
        "strategy": "depth_first",
    },
    # Pattern: Compare → Item A + Item B + Criteria + Use cases + Recommendations
    {
        "pattern": "Compare {thing_a} vs {thing_b}",
        "sub_queries": [
            "{thing_a} overview capabilities strengths",
            "{thing_b} overview capabilities strengths",
            "{thing_a} vs {thing_b} performance benchmarks",
            "use cases for {thing_a}",
            "use cases for {thing_b}",
            "expert recommendations {thing_a} {thing_b}",
        ],
        "source_types": ["web", "github"],
        "strategy": "breadth_first",
    },
    # Pattern: State of X → Definition + Metrics + Leaders + Trends + Outlook
    {
        "pattern": "What is the current state of {topic}",
        "sub_queries": [
            "{topic_short} market size statistics 2024",
            "key players leaders in {topic_short}",
            "{topic_short} recent trends developments",
            "challenges facing {topic_short} industry",
            "{topic_short} future outlook predictions",
        ],
        "source_types": ["web", "news"],
        "strategy": "breadth_first",
    },
]

# Dimensional modifiers to create variations
RECENCY_MODIFIERS = ["", "in 2024", "in 2025", "recent", "latest", "emerging"]
SCOPE_MODIFIERS = ["", "globally", "in the US", "in Europe", "in Asia"]


class DecompositionDatasetCollector:
    def __init__(self):
        
        self.out_path = Path("decomposition/decomposition.jsonl")
        os.makedirs(self.out_path.parent, exist_ok=True)
        
    def collect(self):

        samples = []

        log.info("  Generating rule-based decompositions ...")
        rule_samples = self._generate_rule_based()
        samples.extend(rule_samples)
        log.info(f"    → Generated {len(rule_samples)} rule-based samples")


        log.info("  Generating augmented variations ...")
        aug_samples = self._generate_augmented()
        samples.extend(aug_samples)
        log.info(f"    → Generated {len(aug_samples)} augmented samples")

        
        log.info("  Generating negative → positive correction examples ...")
        correction_samples = self._generate_corrections()
        samples.extend(correction_samples)
        log.info(f"    → Generated {len(correction_samples)} correction samples")

        random.shuffle(samples)
        self._write(samples)
        log.info(f"✅ Dataset 4 saved: {len(samples)} samples → {self.out_path}")

    def _generate_rule_based(self) -> list:
        samples = []
        for domain, questions in SEED_QUESTIONS.items():
            for question in questions:
                sample = self._decompose_question(question, domain)
                if sample:
                    samples.append(sample)
        return samples

    def _generate_augmented(self) -> list:
        """Create variations of seed questions with modifiers."""
        samples = []
        for domain, questions in SEED_QUESTIONS.items():
            for question in questions:
                for rec_mod in RECENCY_MODIFIERS[:3]:  # limit combinations
                    for scope_mod in SCOPE_MODIFIERS[:3]:
                        if not rec_mod and not scope_mod:
                            continue  # skip bare question (already in rule-based)
                        modified_q = question.rstrip("?")
                        if rec_mod:
                            modified_q += f" {rec_mod}"
                        if scope_mod:
                            modified_q += f" {scope_mod}"
                        modified_q += "?"
                        sample = self._decompose_question(modified_q, domain)
                        if sample:
                            samples.append(sample)
        return samples

    def _generate_corrections(self) -> list:
        """
        DPO-style: show a bad decomposition, then the correct one.
        Teaches: don't overlap sub-queries, don't be too vague.
        """
        samples = []
        bad_patterns = [
            "tell me everything about {topic}",
            "research {topic} thoroughly",
            "find all information about {topic}",
        ]

        all_questions = [q for qs in SEED_QUESTIONS.values() for q in qs]
        for question in random.sample(all_questions, min(50, len(all_questions))):
            # Show correction: vague → specific
            topic_short = self._extract_topic_short(question)
            bad_decomposition = [random.choice(bad_patterns).format(topic=topic_short)]
            good_decomposition = self._get_sub_queries(question, topic_short)

            correction_prompt = (
                f"The following query decomposition is too vague and unhelpful:\n"
                f"Query: {question}\n"
                f"Bad decomposition: {json.dumps(bad_decomposition)}\n\n"
                f"Provide a better, specific decomposition as JSON."
            )

            output = json.dumps({
                "sub_queries": good_decomposition,
                "source_types": self._get_source_types(question),
                "search_strategy": "breadth_first",
                "reasoning": "Breaking into specific, non-overlapping searchable sub-queries",
            })

            samples.append({
                "dataset": "synthetic_corrections",
                "task": "query_decomposition",
                "system": SYSTEM_PROMPT,
                "messages": [
                    {"role": "user", "content": correction_prompt},
                    {"role": "assistant", "content": output},
                ],
            })
        return samples

    
    # ── Helpers ───────────────────────────────────────────────────────────────

    def _decompose_question(self, question: str, domain: str) -> dict | None:
        topic_short = self._extract_topic_short(question)
        sub_queries = self._get_sub_queries(question, topic_short)
        source_types = self._get_source_types(question)
        domain_hints = {
            "technology": "technical research with code examples preferred",
            "science": "peer-reviewed sources preferred",
            "business": "market data and news preferred",
            "academic": "academic papers preferred",
            "current_events": "recent news and analysis preferred",
        }

        output = json.dumps({
            "sub_queries": sub_queries,
            "source_types": source_types,
            "search_strategy": random.choice(["breadth_first", "depth_first"]),
            "expected_sources": random.randint(3, 8),
            "domain": domain,
            "notes": domain_hints.get(domain, ""),
        }, indent=None)

        return {
            "dataset": "synthetic_decomposition",
            "task": "query_decomposition",
            "system": SYSTEM_PROMPT,
            "messages": [
                {"role": "user", "content": f"Decompose this research question into specific, searchable sub-queries. Return JSON only.\n\nQuestion: {question}"},
                {"role": "assistant", "content": output},
            ],
        }

    def _extract_topic_short(self, question: str) -> str:
        """Extract a 2-4 word topic from a question."""
        stop_words = {"what", "how", "are", "is", "the", "a", "an", "of",
                      "in", "for", "and", "or", "to", "do", "does", "latest",
                      "current", "state", "recent", "compare", "between"}
        words = re.sub(r"[?.,!]", "", question.lower()).split()
        topic_words = [w for w in words if w not in stop_words]
        return " ".join(topic_words[:4]) if topic_words else question[:30]

    def _get_sub_queries(self, question: str, topic_short: str) -> list:
        """Generate specific sub-queries for a question."""
        q_lower = question.lower()

        # Detect question type and apply appropriate decomposition
        if any(w in q_lower for w in ["compare", "vs", "versus", "difference"]):
            return [
                f"{topic_short} overview advantages",
                f"{topic_short} disadvantages limitations",
                f"{topic_short} benchmarks performance comparison",
                f"{topic_short} use cases real world examples",
                f"{topic_short} expert recommendations 2024",
            ]
        elif any(w in q_lower for w in ["how", "mechanism", "work", "process"]):
            return [
                f"{topic_short} how it works explanation",
                f"{topic_short} technical details implementation",
                f"{topic_short} research papers findings",
                f"{topic_short} practical applications examples",
                f"{topic_short} current challenges limitations",
            ]
        elif any(w in q_lower for w in ["latest", "recent", "breakthroughs", "new"]):
            return [
                f"{topic_short} breakthroughs 2024",
                f"{topic_short} research papers 2024 arxiv",
                f"companies organizations leading {topic_short}",
                f"{topic_short} future trends predictions",
                f"{topic_short} funding investment news",
            ]
        elif any(w in q_lower for w in ["state", "current", "status"]):
            return [
                f"{topic_short} market overview statistics",
                f"key players {topic_short} industry leaders",
                f"{topic_short} recent developments news",
                f"{topic_short} challenges problems 2024",
                f"{topic_short} growth predictions outlook",
            ]
        else:
            return [
                f"{topic_short} overview introduction",
                f"{topic_short} latest research 2024",
                f"{topic_short} key findings results",
                f"{topic_short} experts opinions",
                f"{topic_short} future implications",
            ]

    def _get_source_types(self, question: str) -> list:
        q_lower = question.lower()
        sources = ["web"]
        if any(w in q_lower for w in ["research", "study", "paper", "science", "discovery"]):
            sources.append("arxiv")
        if any(w in q_lower for w in ["code", "software", "framework", "library", "programming"]):
            sources.append("github")
        if any(w in q_lower for w in ["market", "company", "industry", "economy", "business"]):
            sources.append("news")
        return list(set(sources))

    def _write(self, samples: list):
        with open(self.out_path, "w") as f:
            for s in samples:
                f.write(json.dumps(s, ensure_ascii=False) + "\n") 

obj = DecompositionDatasetCollector() 
obj.collect()

In [ ]:
from huggingface_hub import HfApi 

api = HfApi() 

api.create_repo(
    repo_id="Rohit-Katkar2003/decomposition-dataset",
    repo_type="dataset",
    exist_ok=True
)

api.upload_folder(
    folder_path="decomposition",
    repo_id="Rohit-Katkar2003/decomposition-dataset",
    repo_type="dataset",
)

CommitInfo(commit_url='https://huggingface.co/datasets/Rohit-Katkar2003/decomposition-dataset/commit/075ff66e29af7ada0c266be42622c3031322788e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='075ff66e29af7ada0c266be42622c3031322788e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Rohit-Katkar2003/decomposition-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Rohit-Katkar2003/decomposition-dataset'), pr_revision=None, pr_num=None)